# Extração — Procedures by Country (Raw -> Bronze)

Objetivo: a partir dos PDFs já carregados na camada **Raw** do MinIO (bucket `raw`, prefixo `dados_brutos/pdf/`), extrair **apenas** as tabelas "Procedures by Country"/"by Location" (procedimentos estéticos por país), estruturar em formato tabular (long/tidy) e gravar o resultado em Parquet na camada **Bronze** — localmente em `dados_processados/bronze/procedures_by_country/` e no MinIO (bucket `bronze`).

Aqui não há limpeza de dados (duplicados, nulos, validação de schema) — isso é responsabilidade da camada Silver. A camada Bronze apenas estrutura o dado bruto que interessa (a tabela país x procedimento) em formato tabular/colunar, descartando o restante do PDF (texto corrido, gráficos, outras tabelas).

## Verificação de viabilidade

Os PDFs da pasta `pdf/` usam layouts diferentes ao longo dos anos:

- **2018 a 2024:** tabela larga "SURGICAL/NON-SURGICAL PROCEDURES BY COUNTRY (ou BY LOCATION)", com países nas colunas e procedimentos nas linhas — **extraível de forma genérica e confiável**.
- **2017:** não existe uma tabela larga país x procedimento; os dados por país aparecem como texto corrido, uma página por país — **não extraível** com o parser genérico.
- **2016:** o relatório fragmenta a mesma informação em várias mini-tabelas por página, e os mesmos rótulos de linha (ex.: `Total Face & Head Procedures`) se repetem entre tabelas distintas com **valores conflitantes** para o mesmo país — **não é seguro** extrair de forma automática/genérica (dado ambíguo).

A célula de verificação abaixo roda a extração em todos os PDFs presentes na camada raw e reporta, com evidência (conflitos detectados, linhas extraídas), quais são compatíveis. Apenas os compatíveis seguem para a gravação em Bronze.

## Imports

In [1]:
import io
import os
import re
from pathlib import Path

import pandas as pd
import pdfplumber
from dotenv import load_dotenv
from minio import Minio

## Configuração

Define a origem (bucket `raw`, prefixo `dados_brutos/pdf/` — a camada Raw gravada por [extracao_raw.ipynb](extracao_raw.ipynb)) e o destino (bucket `bronze` no MinIO + pasta local `dados_processados/bronze/procedures_by_country/`). O bucket `bronze` é criado aqui se não existir, pelo mesmo motivo de idempotência do notebook Raw: o notebook pode ser reexecutado (ex.: depois de adicionar um novo PDF de survey) sem exigir setup manual prévio.

Gravar em dois lugares (local e MinIO) não é redundância por acaso: o Parquet local serve para inspeção rápida e para os notebooks de EDA/tratamento rodarem sem depender de rede; o Parquet no MinIO é o que efetivamente representa a camada Bronze do data lake, consumível por qualquer outra ferramenta/processo.

In [2]:
load_dotenv(Path.cwd().parent / ".env")

MINIO_ENDPOINT = os.getenv("MINIO_ENDPOINT", "localhost:9000")
MINIO_ACCESS_KEY = os.getenv("MINIO_ACCESS_KEY")
MINIO_SECRET_KEY = os.getenv("MINIO_SECRET_KEY")

BUCKET_RAW = os.getenv("BUCKET_RAW", "raw")
RAW_PDF_PREFIX = "dados_brutos/pdf/"

BUCKET_BRONZE = os.getenv("BUCKET_BRONZE", "bronze")
BRONZE_PREFIX = "procedures_by_country/"

BRONZE_DIR = Path.cwd().parent / "dados_processados" / "bronze" / "procedures_by_country"
BRONZE_DIR.mkdir(parents=True, exist_ok=True)

client = Minio(MINIO_ENDPOINT, access_key=MINIO_ACCESS_KEY, secret_key=MINIO_SECRET_KEY, secure=False)
if not client.bucket_exists(BUCKET_BRONZE):
    client.make_bucket(BUCKET_BRONZE)
    print(f"Bucket '{BUCKET_BRONZE}' criado.")

print(f"MinIO: {MINIO_ENDPOINT} | bucket raw: {BUCKET_RAW} | bucket bronze: {BUCKET_BRONZE}")
print(f"Saida parquet local: {BRONZE_DIR}")

MinIO: localhost:9000 | bucket raw: raw | bucket bronze: bronze
Saida parquet local: C:\Projeto_AI\dados_processados\bronze\procedures_by_country


## Funções de extração

As tabelas "Procedures by Country" seguem o padrão: primeira coluna = procedimento, primeira linha = países. Linhas com todos os valores vazios são cabeçalhos de categoria (ex.: `FACE & HEAD`, `INJECTABLES`) que se aplicam às linhas seguintes até a próxima categoria.

**Por que esse conjunto específico de funções:**
- `find_procedure_pages` localiza a página certa pelo **título**, não pela posição no PDF — a posição varia de relatório para relatório, mas o título "SURGICAL/NON-SURGICAL PROCEDURES BY COUNTRY/LOCATION" é estável entre edições. Os filtros `"RANKING" not in` e `"GROUP" not in` existem porque o ISAPS também publica páginas de ranking e de agrupamento com títulos parecidos, mas em outro formato de tabela — sem esse filtro, o parser tentaria (e falharia, ou pior, extrairia errado) essas páginas também.
- `extract_page` trata linhas totalmente vazias como cabeçalho de categoria (`current_category`) em vez de descartá-las, porque é assim que o PDF representa visualmente a hierarquia categoria → procedimento: não há uma coluna explícita de categoria na tabela, só a posição relativa das linhas.
- `parse_number` normaliza os símbolos que o ISAPS usa para "sem dado" (`-`, `DNA`, `N/A`) para `None`, em vez de deixá-los virar texto numa coluna que deveria ser numérica — sem isso, a coluna `quantidade` ficaria com tipo misto (texto e número) e nenhuma agregação funcionaria.
- `EXCLUDED_LABELS` remove linhas que tecnicamente aparecem na mesma tabela mas não são procedimentos (`POPULATION`, contagem de cirurgiões) — mantê-las misturaria métricas de naturezas completamente diferentes na mesma coluna `quantidade`.

In [3]:
EXCLUDED_LABELS = {"POPULATION", "ESTIMATED NUMBER OF PLASTIC SURGEONS", "PERCENTAGE OF POPULATION"}


def clean_header_cell(c):
    if c is None:
        return ""
    c = str(c).replace("\n", " ").strip()
    c = re.sub(r"\*+$", "", c).strip()
    return c


def clean_country_cell(c):
    # corrige quebras de hifenização de nomes de país partidos em duas linhas pelo PDF
    c = clean_header_cell(c)
    c = re.sub(r"(\w)-\s+(\w)", r"\1\2", c)
    return c


def parse_number(v):
    if v is None:
        return None
    s = str(v).replace("\n", " ").strip()
    if s in ("", "-", "\ufffd", "dna", "DNA", "N/A", "NA"):
        return None
    s = s.replace(",", "")
    try:
        return float(s)
    except ValueError:
        return None


def find_procedure_pages(pdf):
    """Localiza páginas cujo título é 'SURGICAL/NON-SURGICAL PROCEDURES BY COUNTRY|LOCATION'."""
    pages = []
    for i, page in enumerate(pdf.pages):
        text = page.extract_text() or ""
        first_line = text.splitlines()[0].upper() if text else ""
        if (
            "PROCEDURES BY" in first_line
            and ("COUNTRY" in first_line or "LOCATION" in first_line)
            and "RANKING" not in first_line
            and "GROUP" not in first_line
        ):
            pages.append(i)
    return pages


def extract_page(page, tipo):
    """Extrai uma página no formato pais(coluna) x procedimento(linha) para long/tidy."""
    tables = page.extract_tables()
    if not tables:
        return None

    header_table, header = None, None
    for t in tables:
        if not t or not t[0]:
            continue
        h = [clean_header_cell(c) for c in t[0]]
        if h and "PROCEDURES" in h[0].upper() and len(h) > 4:
            header_table, header = t, h
            break
    if header is None:
        return None

    countries = [clean_country_cell(c) for c in header_table[0][1:]]
    rows = []
    current_category = None
    for t in tables:
        for row in t:
            if row == header_table[0] or len(row) != len(header):
                continue
            label = clean_header_cell(row[0])
            if not label or label.upper() in EXCLUDED_LABELS:
                continue
            values = row[1:]
            if not any(v not in (None, "") for v in values):
                current_category = label
                continue
            for country, val in zip(countries, values):
                country_norm = re.sub(r"\s+", "", country or "").upper()
                if not country or country_norm.startswith("WORLDWIDE") or country_norm.startswith("WORLD-WIDE"):
                    continue  # agregado mundial não é um pais
                rows.append({
                    "tipo_procedimento": tipo,
                    "categoria": current_category,
                    "procedimento": label,
                    "pais": country,
                    "quantidade": parse_number(val),
                })
    return pd.DataFrame(rows)


def extract_year(pdf):
    for page in pdf.pages[:6]:
        text = (page.extract_text() or "").upper()
        m = re.search(r"PERFORMED IN (\d{4})", text)
        if m:
            return int(m.group(1))
    return None


def extract_pdf(pdf_bytes):
    with pdfplumber.open(io.BytesIO(pdf_bytes)) as pdf:
        ano = extract_year(pdf)
        page_idxs = find_procedure_pages(pdf)
        dfs = []
        for i in page_idxs:
            title = (pdf.pages[i].extract_text() or "").splitlines()[0].upper()
            tipo = "NAO-CIRURGICO" if "NON" in title else "CIRURGICO"
            df = extract_page(pdf.pages[i], tipo)
            if df is not None and not df.empty:
                dfs.append(df)
    cols = ["tipo_procedimento", "categoria", "procedimento", "pais", "quantidade"]
    full = pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame(columns=cols)
    return full, ano, page_idxs

## Verificação de viabilidade (todos os PDFs da camada raw)

Roda a extração em **todos** os PDFs do prefixo raw e classifica cada um como `COMPATIVEL`/`INCOMPATIVEL` com evidência, em vez de assumir que todo PDF segue o mesmo layout. O critério de incompatibilidade tem duas partes:

1. **Nenhuma página localizada** (`page_idxs` vazio) — o relatório daquele ano não tem a tabela larga país x procedimento no formato esperado.
2. **Conflito de valores** — a mesma combinação `tipo_procedimento`/`categoria`/`procedimento`/`pais` aparece mais de uma vez com `quantidade` diferente. Isso indica que o PDF fragmentou a mesma informação em várias mini-tabelas (como acontece no relatório de 2016), e não há como saber automaticamente qual dos valores conflitantes está correto — extrair de qualquer forma arriscaria gravar um número errado sem nenhum aviso.

Só os arquivos `COMPATIVEL` seguem para `tabelas_bronze` e, depois, para o Parquet — os demais são descartados aqui, de forma explícita e documentada no `relatorio_viabilidade`, em vez de causarem um erro silencioso mais adiante.

In [4]:
KEY_COLS = ["tipo_procedimento", "categoria", "procedimento", "pais"]
COLS_FINAIS = ["arquivo_origem", "ano_referencia", "tipo_procedimento", "categoria", "procedimento", "pais", "quantidade"]

report_rows = []
tabelas_bronze = {}

objects = list(client.list_objects(BUCKET_RAW, prefix=RAW_PDF_PREFIX, recursive=True))
for obj in objects:
    fname = obj.object_name.rsplit("/", 1)[-1]
    pdf_bytes = client.get_object(BUCKET_RAW, obj.object_name).read()
    df, ano, page_idxs = extract_pdf(pdf_bytes)

    conflitantes = 0
    if not df.empty:
        # usa um sentinela no lugar de NaN: groupby por padrao descarta grupos com chave nula
        chaves = df[KEY_COLS].fillna("__NA__")
        conflitantes = int((df.groupby([chaves[c] for c in KEY_COLS])["quantidade"].nunique() > 1).sum())

    if not page_idxs or df.empty:
        status, motivo = "INCOMPATIVEL", "nenhuma tabela larga (pais x procedimento) foi localizada neste layout"
    elif conflitantes > 0:
        status, motivo = "INCOMPATIVEL", f"{conflitantes} combinacoes pais/procedimento com valores conflitantes entre tabelas (layout fragmentado)"
    else:
        status, motivo = "COMPATIVEL", "tabela pais x procedimento extraida com sucesso"

    report_rows.append({
        "arquivo": fname, "ano_referencia": ano, "paginas_candidatas": len(page_idxs),
        "linhas_extraidas": len(df), "status": status, "motivo": motivo,
    })
    if status == "COMPATIVEL":
        df = df.copy()
        df["ano_referencia"] = ano
        df["arquivo_origem"] = fname
        tabelas_bronze[fname] = df[COLS_FINAIS].reset_index(drop=True)

relatorio_viabilidade = pd.DataFrame(report_rows).sort_values("arquivo").reset_index(drop=True)
relatorio_viabilidade

,arquivo,ano_referencia,paginas_candidatas,linhas_extraidas,status,motivo
0,global-survey-full-report-2019-english.pdf,2019,3,640,COMPATIVEL,tabela pais x procedimento extraida com sucesso
1,isaps-global-survey-2024.pdf,2024,9,1568,COMPATIVEL,tabela pais x procedimento extraida com sucesso
2,isaps-global-survey-results-2018-1.pdf,2018,3,390,COMPATIVEL,tabela pais x procedimento extraida com sucesso
3,isaps-global-survey_2020.pdf,2020,3,560,COMPATIVEL,tabela pais x procedimento extraida com sucesso
4,isaps-global-survey_2021.pdf,2021,3,588,COMPATIVEL,tabela pais x procedimento extraida com sucesso
5,isaps-global-survey_2022.pdf,2022,3,672,COMPATIVEL,tabela pais x procedimento extraida com sucesso
6,isaps-global-survey_2023.pdf,2023,5,966,COMPATIVEL,tabela pais x procedimento extraida com sucesso


## Prévia das tabelas em formato Bronze (uma por PDF, sem tratamento/limpeza)

Mostra as primeiras linhas de cada tabela extraída antes de gravar qualquer coisa em disco — uma conferência visual rápida de que a extração fez sentido (colunas certas, países reconhecíveis, valores numéricos plausíveis) antes de confiar no resultado.

In [5]:
for fname, df in tabelas_bronze.items():
    print(f"\n=== {fname} ({len(df)} linhas, {df['pais'].nunique()} paises) ===")
    display(df.head(5))


=== global-survey-full-report-2019-english.pdf (640 linhas, 16 paises) ===


,arquivo_origem,ano_referencia,tipo_procedimento,categoria,procedimento,pais,quantidade
0,global-survey-full-report-2019-english.pdf,2019,CIRURGICO,FACE & HEAD,Brow Lift,USA,24219.0
1,global-survey-full-report-2019-english.pdf,2019,CIRURGICO,FACE & HEAD,Brow Lift,BRAZIL,40214.0
2,global-survey-full-report-2019-english.pdf,2019,CIRURGICO,FACE & HEAD,Brow Lift,JAPAN,200.0
3,global-survey-full-report-2019-english.pdf,2019,CIRURGICO,FACE & HEAD,Brow Lift,MEXICO,17034.0
4,global-survey-full-report-2019-english.pdf,2019,CIRURGICO,FACE & HEAD,Brow Lift,ITALY,9132.0



=== isaps-global-survey-2024.pdf (1568 linhas, 32 paises) ===


,arquivo_origem,ano_referencia,tipo_procedimento,categoria,procedimento,pais,quantidade
0,isaps-global-survey-2024.pdf,2024,CIRURGICO,FACE & HEAD,Brow Lift,ARGENTINA,7205.0
1,isaps-global-survey-2024.pdf,2024,CIRURGICO,FACE & HEAD,Brow Lift,AUSTRALIA,2783.0
2,isaps-global-survey-2024.pdf,2024,CIRURGICO,FACE & HEAD,Brow Lift,BANGLADESH,340.0
3,isaps-global-survey-2024.pdf,2024,CIRURGICO,FACE & HEAD,Brow Lift,BRAZIL,74716.0
4,isaps-global-survey-2024.pdf,2024,CIRURGICO,FACE & HEAD,Brow Lift,CHILE,908.0



=== isaps-global-survey-results-2018-1.pdf (390 linhas, 10 paises) ===


...

,arquivo_origem,ano_referencia,tipo_procedimento,categoria,procedimento,pais,quantidade
0,isaps-global-survey_2021.pdf,2021,CIRURGICO,FACE & HEAD,Brow Lift,USA,41008.0
1,isaps-global-survey_2021.pdf,2021,CIRURGICO,FACE & HEAD,Brow Lift,BRAZIL,40140.0
2,isaps-global-survey_2021.pdf,2021,CIRURGICO,FACE & HEAD,Brow Lift,JAPAN,175.0
3,isaps-global-survey_2021.pdf,2021,CIRURGICO,FACE & HEAD,Brow Lift,MEXICO,15785.0
4,isaps-global-survey_2021.pdf,2021,CIRURGICO,FACE & HEAD,Brow Lift,GERMANY,12493.0



=== isaps-global-survey_2022.pdf (672 linhas, 16 paises) ===


,arquivo_origem,ano_referencia,tipo_procedimento,categoria,procedimento,pais,quantidade
0,isaps-global-survey_2022.pdf,2022,CIRURGICO,FACE & HEAD,Brow Lift,US,25367.0
1,isaps-global-survey_2022.pdf,2022,CIRURGICO,FACE & HEAD,Brow Lift,BRAZIL,54324.0
2,isaps-global-survey_2022.pdf,2022,CIRURGICO,FACE & HEAD,Brow Lift,JAPAN,1456.0
3,isaps-global-survey_2022.pdf,2022,CIRURGICO,FACE & HEAD,Brow Lift,MEXICO,25377.0
4,isaps-global-survey_2022.pdf,2022,CIRURGICO,FACE & HEAD,Brow Lift,TURKEY,16110.0



=== isaps-global-survey_2023.pdf (966 linhas, 23 paises) ===


,arquivo_origem,ano_referencia,tipo_procedimento,categoria,procedimento,pais,quantidade
0,isaps-global-survey_2023.pdf,2023,CIRURGICO,FACE & HEAD,Brow Lift,ARGENTINA,9813.0
1,isaps-global-survey_2023.pdf,2023,CIRURGICO,FACE & HEAD,Brow Lift,BANGLADESH,306.0
2,isaps-global-survey_2023.pdf,2023,CIRURGICO,FACE & HEAD,Brow Lift,BELGIUM,2077.0
3,isaps-global-survey_2023.pdf,2023,CIRURGICO,FACE & HEAD,Brow Lift,BRAZIL,54191.0
4,isaps-global-survey_2023.pdf,2023,CIRURGICO,FACE & HEAD,Brow Lift,COLOMBIA,6284.0


## Conversão para Parquet (camada Bronze)

Cada PDF compatível gera um arquivo Parquet, gravado localmente em `dados_processados/bronze/procedures_by_country/` e também enviado ao MinIO no bucket `bronze`, sob o prefixo `procedures_by_country/`.

**Por que um arquivo por PDF, em vez de um único Parquet consolidado:** preserva a granularidade de origem — cada arquivo representa exatamente uma edição do relatório ISAPS. Isso facilita reprocessar um único ano (se o parser for corrigido ou o PDF daquele ano for atualizado) sem precisar regravar os demais, e deixa explícito no próprio nome do arquivo (`{ano}_{slug}.parquet`) qual relatório originou aquele dado.

In [6]:
arquivos_gerados = []
for fname, df in tabelas_bronze.items():
    ano = df["ano_referencia"].iloc[0]
    slug = re.sub(r"[^a-z0-9]+", "_", Path(fname).stem.lower()).strip("_")
    out_name = f"{ano}_{slug}.parquet"
    out_path = BRONZE_DIR / out_name

    df.to_parquet(out_path, engine="pyarrow", index=False)
    arquivos_gerados.append(out_path)

    object_name = f"{BRONZE_PREFIX}{out_name}"
    client.fput_object(BUCKET_BRONZE, object_name, str(out_path))
    print(f"Gravado: {out_path} ({len(df)} linhas) -> s3://{BUCKET_BRONZE}/{object_name}")

Gravado: C:\Projeto_AI\dados_processados\bronze\procedures_by_country\2019_global_survey_full_report_2019_english.parquet (640 linhas) -> s3://bronze/procedures_by_country/2019_global_survey_full_report_2019_english.parquet
Gravado: C:\Projeto_AI\dados_processados\bronze\procedures_by_country\2024_isaps_global_survey_2024.parquet (1568 linhas) -> s3://bronze/procedures_by_country/2024_isaps_global_survey_2024.parquet
Gravado: C:\Projeto_AI\dados_processados\bronze\procedures_by_country\2018_isaps_global_survey_results_2018_1.parquet (390 linhas) -> s3://bronze/procedures_by_country/2018_isaps_global_survey_results_2018_1.parquet
Gravado: C:\Projeto_AI\dados_processados\bronze\procedures_by_country\2020_isaps_global_survey_2020.parquet (560 linhas) -> s3://bronze/procedures_by_country/2020_isaps_global_survey_2020.parquet


Gravado: C:\Projeto_AI\dados_processados\bronze\procedures_by_country\2021_isaps_global_survey_2021.parquet (588 linhas) -> s3://bronze/procedures_by_country/2021_isaps_global_survey_2021.parquet
Gravado: C:\Projeto_AI\dados_processados\bronze\procedures_by_country\2022_isaps_global_survey_2022.parquet (672 linhas) -> s3://bronze/procedures_by_country/2022_isaps_global_survey_2022.parquet
Gravado: C:\Projeto_AI\dados_processados\bronze\procedures_by_country\2023_isaps_global_survey_2023.parquet (966 linhas) -> s3://bronze/procedures_by_country/2023_isaps_global_survey_2023.parquet


## Conferência final

Relê os Parquet recém-gravados do disco (em vez de reaproveitar o DataFrame em memória) para confirmar que o que foi persistido é exatamente o que a etapa de gravação pretendia salvar — um mesmo tipo de checagem de "ida e volta" (write, depois read) que também aparece na conferência final do notebook Raw e do notebook Silver.

In [7]:
# Le os parquets gerados localmente e mostra um resumo consolidado
consolidado = pd.concat([pd.read_parquet(p) for p in arquivos_gerados], ignore_index=True)
print(f"Total consolidado: {len(consolidado)} linhas, {consolidado['arquivo_origem'].nunique()} arquivos, anos {sorted(consolidado['ano_referencia'].unique())}")
display(consolidado.sample(10, random_state=42))

Total consolidado: 5384 linhas, 7 arquivos, anos [np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]


,arquivo_origem,ano_referencia,tipo_procedimento,categoria,procedimento,pais,quantidade
3997,isaps-global-survey_2022.pdf,2022,CIRURGICO,BREAST,Total Breast Procedures,GREECE,23357.0
1717,isaps-global-survey-2024.pdf,2024,CIRURGICO,BODY & EXTREMITIES,Other Outer Genital Surgery,UK,510.0
1782,isaps-global-survey-2024.pdf,2024,NAO-CIRURGICO,FACIAL REJUVENATION,Full Field Ablative,COLOMBIA,6240.0
3127,isaps-global-survey_2020.pdf,2020,NAO-CIRURGICO,OTHER,Total Other Procedures,GREECE,105501.0
248,global-survey-full-report-2019-english.pdf,2019,CIRURGICO,BODY & EXTREMITIES,Abdominoplasty,INDIA,29088.0
3392,isaps-global-survey_2021.pdf,2021,CIRURGICO,BODY & EXTREMITIES,Abdominoplasty,SPAIN,11520.0
3782,isaps-global-survey_2022.pdf,2022,CIRURGICO,FACE & HEAD,Eyelid Surgery,TURKEY,42630.0
3121,isaps-global-survey_2020.pdf,2020,NAO-CIRURGICO,OTHER,Total Other Procedures,MEXICO,52050.0
3624,isaps-global-survey_2021.pdf,2021,NAO-CIRURGICO,FACIAL REJUVENATION,Full Field Ablative,GERMANY,4604.0
1666,isaps-global-survey-2024.pdf,2024,CIRURGICO,BREAST,Total Breast Procedures,US,542150.0
